# Projeto Semestral 1 — Ciência de Dados
## 04832 — Percepção dos Brasileiros Acerca da Democracia

**Objetivo deste notebook:** carregar, estruturar, limpar e preparar as duas bases do projeto (CESOP e TSE) para as análises posteriores. Aqui não há análise exploratória — apenas o pipeline de preparação dos dados.

Ao final, este notebook gera os seguintes arquivos em `data/processed/`:
- `cesop_clean.parquet` — base CESOP tratada
- `tse_clean.parquet` — base TSE tratada (linhas-detalhe)
- `tse_uf.parquet` — TSE agregado por UF
- `tse_perfil.parquet` — TSE agregado por perfil demográfico

In [1]:
# Importações e definição de caminhos constantes.
# Centralizar os paths aqui evita reescrever strings longas ao longo do notebook
# e facilita a portabilidade para o Google Colab (basta ajustar a raiz).
import os
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadstat

# Raiz do projeto a partir da pasta notebooks/
PROJECT_ROOT = Path("..").resolve()

# Paths das bases brutas
PATH_CESOP_SAV = PROJECT_ROOT / "data" / "raw" / "cesop" / "04832.SAV"
PATH_TSE_CSV = PROJECT_ROOT / "data" / "raw" / "tse" / "perfil_comparecimento_abstencao_2022" / "perfil_comparecimento_abstencao_2022_BRASIL.csv"

# Pasta de saída dos dados tratados (criada se não existir)
PATH_PROCESSED = PROJECT_ROOT / "data" / "processed"
PATH_PROCESSED.mkdir(parents=True, exist_ok=True)

print("CESOP SAV existe?", PATH_CESOP_SAV.exists())
print("TSE CSV existe?", PATH_TSE_CSV.exists())

CESOP SAV existe? True
TSE CSV existe? True


In [2]:
# Leitura da base SPSS do CESOP.
# pyreadstat retorna o DataFrame e um objeto meta com rótulos das colunas e
# mapeamentos código→categoria; ambos serão usados nas etapas seguintes.
df_cesop, meta_cesop = pyreadstat.read_sav(str(PATH_CESOP_SAV))
print("Linhas e colunas:", df_cesop.shape)

Linhas e colunas: (2000, 27)


In [3]:
# Inspeção rápida dos dtypes e preenchimento.
# Útil para identificar colunas com muitos NaN (P2_2/P2_3 e P3_2..P3_6),
# que correspondem a perguntas com respostas múltiplas em ordem.
df_cesop.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 27 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   SEXO             2000 non-null   float64
 1   IDADE            2000 non-null   float64
 2   FX_ID            2000 non-null   float64
 3   ESCOLARIDADE     2000 non-null   float64
 4   P1A              2000 non-null   float64
 5   P1B              2000 non-null   float64
 6   P1C              2000 non-null   float64
 7   P2_1             2000 non-null   float64
 8   P2_2             1883 non-null   float64
 9   P2_3             1791 non-null   float64
 10  P3_1             2000 non-null   float64
 11  P3_2             818 non-null    float64
 12  P3_3             304 non-null    float64
 13  P3_4             88 non-null     float64
 14  P3_5             55 non-null     float64
 15  P3_6             41 non-null     float64
 16  P4               2000 non-null   float64
 17  RACA          

In [4]:
# Amostra das primeiras linhas com os códigos numéricos originais do SPSS.
# Antes da recodificação, todos os valores são floats numéricos (1.0, 2.0, ...).
df_cesop.head()

,SEXO,IDADE,FX_ID,ESCOLARIDADE,P1A,P1B,P1C,P2_1,P2_2,P2_3,...,RACA,RELIGIAO,REND1,REND2,REGIAO,COND,PORTE,ID_Ipec,DATA_ENTREVISTA,TIPO_COLETA
0,2.0,29.0,3.0,16.0,1.0,1.0,1.0,1.0,11.0,9.0,...,1.0,10.0,4.0,2.0,2.0,1.0,7.0,189519187.0,2023-09-01,1.0
1,1.0,77.0,7.0,4.0,2.0,2.0,2.0,11.0,9.0,10.0,...,3.0,1.0,4.0,4.0,5.0,1.0,7.0,189525808.0,2023-09-01,1.0
2,2.0,46.0,5.0,14.0,2.0,2.0,2.0,1.0,9.0,5.0,...,3.0,10.0,6.0,6.0,4.0,2.0,6.0,189526277.0,2023-09-01,1.0
3,1.0,72.0,7.0,6.0,2.0,2.0,2.0,1.0,2.0,4.0,...,3.0,1.0,6.0,6.0,3.0,3.0,4.0,189532152.0,2023-09-01,1.0
4,2.0,53.0,5.0,16.0,2.0,2.0,2.0,4.0,9.0,10.0,...,3.0,11.0,98.0,6.0,3.0,3.0,4.0,189532154.0,2023-09-01,1.0


In [5]:
# Rótulos originais das colunas no SPSS, usados como dicionário descritivo.
# Salvamos como dict para referência rápida durante o tratamento.
labels_cesop = dict(zip(meta_cesop.column_names, meta_cesop.column_labels))
labels_cesop

{'SEXO': None,
 'IDADE': None,
 'FX_ID': 'Faixas de idade',
 'ESCOLARIDADE': None,
 'P1A': 'P.01A) Você se lembra em quem votou para deputado(a) estadual nas eleições gerais de 2022?',
 'P1B': 'P.01B) Você se lembra em quem votou para deputado(a) federal nas eleições gerais de 2022?',
 'P1C': 'P.01C) Você se lembra em quem votou para senador(a) nas eleições gerais de 2022?',
 'P2_1': 'P.02) Qual destas propostas você acha que deveria ser prioridade de um(a) político(a)? (1º lugar)',
 'P2_2': 'P.02) Qual destas propostas você acha que deveria ser prioridade de um(a) político(a)? (2º lugar)',
 'P2_3': 'P.02) Qual destas propostas você acha que deveria ser prioridade de um(a) político(a)? (3º lugar)',
 'P3_1': 'P.03) Algumas pessoas dizem que a divulgação de fake news - notícias ou conteúdos falsos podem prejudicar a democracia. Quais dessas opções você acredita que poderiam contribuir no combate à divulgação de fake news?',
 'P3_2': 'P.03) Algumas pessoas dizem que a divulgação de fake n

In [6]:
# Mapeamentos código→categoria embarcados no arquivo SPSS.
# Nem todas as variáveis têm value labels — variáveis numéricas livres (IDADE, ID_Ipec)
# aparecem sem entrada aqui.
list(meta_cesop.variable_value_labels.keys())

['SEXO',
 'FX_ID',
 'ESCOLARIDADE',
 'P1A',
 'P1B',
 'P1C',
 'P2_1',
 'P2_2',
 'P2_3',
 'P3_1',
 'P3_2',
 'P3_3',
 'P3_4',
 'P3_5',
 'P3_6',
 'P4',
 'RACA',
 'RELIGIAO',
 'REND1',
 'REND2',
 'REGIAO',
 'COND',
 'PORTE',
 'TIPO_COLETA']

## 1. CESOP — Estruturação dos dados

Padronizamos os nomes das variáveis de pergunta (`P1A` → `P_01A`, etc.) para alinhar com o dicionário de dados, e convertemos `DATA_ENTREVISTA` para `datetime`. Nomes consistentes facilitam o cruzamento e a documentação posterior.

In [7]:
# Renomeia as variáveis de pergunta para o padrão do dicionário de dados (P_01A em vez de P1A).
# Consistência com a documentação evita confusão durante a análise.
rename_map = {
    'P1A': 'P_01A', 'P1B': 'P_01B', 'P1C': 'P_01C',
    'P2_1': 'P_02_1', 'P2_2': 'P_02_2', 'P2_3': 'P_02_3',
    'P3_1': 'P_03_1', 'P3_2': 'P_03_2', 'P3_3': 'P_03_3',
    'P3_4': 'P_03_4', 'P3_5': 'P_03_5', 'P3_6': 'P_03_6',
    'P4':   'P_04',
}
df_cesop = df_cesop.rename(columns=rename_map)

# DATA_ENTREVISTA chega como string ('YYYY-MM-DD'); converter para datetime
# permite filtros temporais e ordenações nos passos seguintes.
df_cesop['DATA_ENTREVISTA'] = pd.to_datetime(df_cesop['DATA_ENTREVISTA'], errors='coerce')

print("Colunas renomeadas:", [v for v in rename_map.values() if v in df_cesop.columns])
print("Dtype DATA_ENTREVISTA:", df_cesop['DATA_ENTREVISTA'].dtype)

Colunas renomeadas: ['P_01A', 'P_01B', 'P_01C', 'P_02_1', 'P_02_2', 'P_02_3', 'P_03_1', 'P_03_2', 'P_03_3', 'P_03_4', 'P_03_5', 'P_03_6', 'P_04']
Dtype DATA_ENTREVISTA: datetime64[ns]


## 2. CESOP — Limpeza de valores especiais

Códigos especiais do SPSS representam ausência de resposta e não devem ser confundidos com categorias válidas:

- `99` — "Não respondeu / Não sabe" (presente em quase todas as perguntas)
- `98` — em `REND1`, significa "Não tem rendimento pessoal" e é **categoria válida** (não vira NaN)

Após a substituição, aplicamos os `variable_value_labels` para que cada coluna categórica fique com textos legíveis no lugar dos códigos numéricos.

In [8]:
# Mapeamento explícito dos códigos de não-resposta para NaN.
# Manter a tabela aqui (em vez de aplicar globalmente "99 → NaN") evita corromper
# REND1, em que 98 = "Não tem rendimento pessoal" é categoria VÁLIDA, não missing.
codigos_nan_por_coluna = {
    'RELIGIAO': [99],
    'REND1':    [99],
    'REND2':    [99],
    'P_01A':    [99], 'P_01B': [99], 'P_01C': [99],
    'P_02_1':   [99], 'P_02_2': [99], 'P_02_3': [99],
    'P_03_1':   [99], 'P_03_2': [99], 'P_03_3': [99],
    'P_03_4':   [99], 'P_03_5': [99], 'P_03_6': [99],
    'P_04':     [99],
}

# Conta NaN antes e depois para ter visibilidade do efeito da substituição.
antes = df_cesop[list(codigos_nan_por_coluna.keys())].isna().sum()
for col, codigos in codigos_nan_por_coluna.items():
    df_cesop[col] = df_cesop[col].replace(codigos, np.nan)
depois = df_cesop[list(codigos_nan_por_coluna.keys())].isna().sum()

pd.DataFrame({'NaN_antes': antes, 'NaN_depois': depois, 'delta': depois - antes})

,NaN_antes,NaN_depois,delta
RELIGIAO,0,29,29
REND1,0,77,77
REND2,0,110,110
P_01A,0,12,12
P_01B,0,10,10
P_01C,0,12,12
P_02_1,0,44,44
P_02_2,117,117,0
P_02_3,209,209,0
P_03_1,0,201,201


In [9]:
# Substitui códigos numéricos por rótulos legíveis em cada coluna categórica.
# Atualizamos as chaves do dicionário SPSS para refletir os nomes renomeados (P1A → P_01A).
# Colunas sem value labels (IDADE, ID_Ipec, DATA_ENTREVISTA, TIPO_COLETA) ficam inalteradas.
value_labels = {}
for col_original, mapa in meta_cesop.variable_value_labels.items():
    col_atual = rename_map.get(col_original, col_original)
    if col_atual in df_cesop.columns:
        value_labels[col_atual] = mapa

# .map() preserva os NaN inseridos no passo anterior.
# Convertemos para category — economiza memória e habilita ordenação/plotagem categorial.
for col, mapa in value_labels.items():
    df_cesop[col] = df_cesop[col].map(mapa).astype('category')

df_cesop.head()

,SEXO,IDADE,FX_ID,ESCOLARIDADE,P_01A,P_01B,P_01C,P_02_1,P_02_2,P_02_3,...,RACA,RELIGIAO,REND1,REND2,REGIAO,COND,PORTE,ID_Ipec,DATA_ENTREVISTA,TIPO_COLETA
0,FEM,29.0,25 A 34,Superior completo,Sim,Sim,Sim,Reduzir as desigualdades sociais,Reduzir a violência,Melhorar a qualidade da saúde,...,Branca,Outras Evangélicas específicas,MAIS DE 2 A 5,MAIS DE 10 A 20,NORDESTE,CAPITAL,MAIS DE 500.000,189519187.0,2023-09-01,Face a face
1,MAS,77.0,65 E MAIS,1ª série (ou 2º ano),Não,Não,Não,Reduzir a violência,Melhorar a qualidade da saúde,Melhorar a qualidade da educação,...,Parda,Católica Apostólica Romana,MAIS DE 2 A 5,MAIS DE 2 A 5,CENTRO OESTE,CAPITAL,MAIS DE 500.000,189525808.0,2023-09-01,Face a face
2,FEM,46.0,45 A 54,3ª série,Não,Não,Não,Reduzir as desigualdades sociais,Melhorar a qualidade da saúde,Combater as mudanças climáticas/desmatamento,...,Parda,Outras Evangélicas específicas,ATÉ 1,ATÉ 1,SUL,PERIFERIA,DE 100.001 A 500.000,189526277.0,2023-09-01,Face a face
3,MAS,72.0,65 E MAIS,3ª série (ou 4º ano),Não,Não,Não,Reduzir as desigualdades sociais,"Combater o preconceito (racismo, homofobia, di...",Incentivar a geração de empregos,...,Parda,Católica Apostólica Romana,ATÉ 1,ATÉ 1,SUDESTE,INTERIOR,DE 20.001 A 50.000,189532152.0,2023-09-01,Face a face
4,FEM,53.0,45 A 54,Superior completo,Não,Não,Não,Incentivar a geração de empregos,Melhorar a qualidade da saúde,Melhorar a qualidade da educação,...,Parda,Evangélica - Não sabe especificar,NÃO TEM RENDIMENTO PESSOAL,ATÉ 1,SUDESTE,INTERIOR,DE 20.001 A 50.000,189532154.0,2023-09-01,Face a face


## 3. CESOP — Variáveis derivadas

Criamos agrupamentos auxiliares para reduzir a granularidade de variáveis com muitas categorias. Isso simplifica visualizações e cruzamentos posteriores:

- `ESCOL_GRUPO`: agrupa os 16 níveis de escolaridade em **5 grupos** (Analfabeto / Sabe ler e escrever sem escola / Até fundamental / Médio / Superior)
- `RENDA_GRUPO`: agrupa as faixas de `REND1` em **4 grupos** (Até 1SM / 1–5SM / Acima de 5SM / Acima de 20SM)
- `FX_ID`: reforçamos a ordem ordinal (faixas etárias crescentes, com os labels exatos do SPSS)

In [10]:
# ESCOL_GRUPO: agrega os 16 níveis de escolaridade em 5 grandes grupos.
# Usamos os labels textuais (já aplicados acima), não os códigos numéricos.

escol_analfabeto = {
    'Analfabeto'
}

escol_alfabetizado = {
    'Sabe ler/ escrever, mas não cursou escola'
}

escol_fundamental = {
    'Pré-escola (ou 1º ano)', '1ª série (ou 2º ano)', '2ª série (ou 3º ano)',
    '3ª série (ou 4º ano)', '4ª série (ou 5º ano)', '5ª série (ou 6º ano)',
    '6ª série (ou 7º ano)', '7ª série (ou 8º ano)', '8ª série (ou 9º ano)',
}

escol_medio = {'1ª série', '2ª série', '3ª série'}

escol_superior = {'Superior incompleto', 'Superior completo'}

def _classificar_escol(valor):
    if valor in escol_analfabeto:
        return 'Analfabeto'
    if valor in escol_alfabetizado:
        return 'Sabe ler/escrever, mas não cursou escola'
    if valor in escol_fundamental:
        return 'Até fundamental'
    if valor in escol_medio:
        return 'Médio'
    if valor in escol_superior:
        return 'Superior'
    return np.nan

df_cesop['ESCOL_GRUPO'] = pd.Categorical(
    df_cesop['ESCOLARIDADE'].astype(object).map(_classificar_escol),
    categories=['Analfabeto', 'Sabe ler/escrever, mas não cursou escola', 'Até fundamental', 'Médio', 'Superior'],
    ordered=True,
)

# RENDA_GRUPO: agrega as faixas de REND1 em 4 grupos.
# Combinamos os extremos para reduzir esparsidade nas tabelas cruzadas.
renda_ate_1sm = {'ATÉ 1', 'NÃO TEM RENDIMENTO PESSOAL'}

renda_1_5sm = {'MAIS DE 1 A 2', 'MAIS DE 2 A 5'}

renda_acima_5sm = {
    'MAIS DE 5 A 10',
    'MAIS DE 10 A 20',
}

renda_acima_20 = {'MAIS DE 20'}

def _classificar_renda(valor):
    if valor in renda_ate_1sm:
        return 'Até 1 salário mínimo'
    if valor in renda_1_5sm:
        return '1 a 5 salários mínimos'
    if valor in renda_acima_5sm:
        return 'Acima de 5 salários mínimos'
    if valor in renda_acima_20:
        return 'Acima de 20 salários mínimos'
    return np.nan

df_cesop['RENDA_GRUPO'] = pd.Categorical(
    df_cesop['REND1'].astype(object).map(_classificar_renda),
    categories=['Até 1 salário mínimo', '1 a 5 salários mínimos', 'Acima de 5 salários mínimos', 'Acima de 20 salários mínimos'],
    ordered=True,
)

df_cesop[['ESCOLARIDADE', 'ESCOL_GRUPO', 'REND1', 'RENDA_GRUPO']].head(10)

,ESCOLARIDADE,ESCOL_GRUPO,REND1,RENDA_GRUPO
0,Superior completo,Superior,MAIS DE 2 A 5,1 a 5 salários mínimos
1,1ª série (ou 2º ano),Até fundamental,MAIS DE 2 A 5,1 a 5 salários mínimos
2,3ª série,Médio,ATÉ 1,Até 1 salário mínimo
3,3ª série (ou 4º ano),Até fundamental,ATÉ 1,Até 1 salário mínimo
4,Superior completo,Superior,NÃO TEM RENDIMENTO PESSOAL,Até 1 salário mínimo
5,4ª série (ou 5º ano),Até fundamental,MAIS DE 1 A 2,1 a 5 salários mínimos
6,4ª série (ou 5º ano),Até fundamental,MAIS DE 2 A 5,1 a 5 salários mínimos
7,3ª série,Médio,MAIS DE 1 A 2,1 a 5 salários mínimos
8,Superior completo,Superior,MAIS DE 2 A 5,1 a 5 salários mínimos
9,3ª série,Médio,ATÉ 1,Até 1 salário mínimo


## 4. CESOP — Validação e salvamento

Conferimos o resultado do tratamento (dtypes, valores ausentes) e salvamos a base limpa em formato Parquet. O Parquet preserva dtypes categóricos e reduz drasticamente o tamanho do arquivo em disco.

In [11]:
# Resumo final dos dtypes e completude após tratamento.
print("=== Dtypes ===")
print(df_cesop.dtypes)
print("\n=== Total de NaN por coluna (top 20) ===")
print(df_cesop.isna().sum().sort_values(ascending=False).head(20))

=== Dtypes ===
SEXO                     category
IDADE                     float64
FX_ID                    category
ESCOLARIDADE             category
P_01A                    category
P_01B                    category
P_01C                    category
P_02_1                   category
P_02_2                   category
P_02_3                   category
P_03_1                   category
P_03_2                   category
P_03_3                   category
P_03_4                   category
P_03_5                   category
P_03_6                   category
P_04                     category
RACA                     category
RELIGIAO                 category
REND1                    category
REND2                    category
REGIAO                   category
COND                     category
PORTE                    category
ID_Ipec                   float64
DATA_ENTREVISTA    datetime64[ns]
TIPO_COLETA              category
ESCOL_GRUPO              category
RENDA_GRUPO              category

In [12]:
# Salva a base CESOP tratada em formato Parquet.
# Parquet preserva dtypes categóricos sem precisar de recodificação extra ao reler.
path_out_cesop = PATH_PROCESSED / 'cesop_clean.parquet'
df_cesop.to_parquet(path_out_cesop, index=False)
print(f"Salvo: {path_out_cesop} ({path_out_cesop.stat().st_size / 1024:.1f} KB)")

Salvo: C:\Users\edfeh\OneDrive\Área de Trabalho\Estudos\IMT\5 Ano\Ciência de Dados\Projeto_EDA_Democracia\data\processed\cesop_clean.parquet (55.8 KB)


## 5. TSE — Carregamento otimizado

O arquivo `perfil_comparecimento_abstencao_2022_BRASIL.csv` tem cerca de 2.3 GB. Carregamos apenas as colunas relevantes para o cruzamento com a CESOP (perfil demográfico + contagens de comparecimento/abstenção), descartando as colunas que não usaremos.

O arquivo usa **separador `;`** e **encoding UTF-8** (confirmado por inspeção do cabeçalho).

In [14]:
# Carregamento parcial do TSE: apenas as colunas necessárias.
# As colunas escolhidas suportam todos os cruzamentos previstos com a CESOP.
colunas_tse = [
    'SG_UF', 'NM_MUNICIPIO',
    'DS_GENERO', 'DS_ESTADO_CIVIL',
    'DS_FAIXA_ETARIA', 'DS_GRAU_ESCOLARIDADE', 'DS_COR_RACA',
    'QT_APTOS', 'QT_COMPARECIMENTO', 'QT_ABSTENCAO',
]

df_tse = pd.read_csv(
    PATH_TSE_CSV,
    sep=';',
    encoding='latin1',
    usecols=colunas_tse,
)
print("Linhas e colunas:", df_tse.shape)
df_tse.head()

Linhas e colunas: (8785738, 10)


,SG_UF,NM_MUNICIPIO,DS_GENERO,DS_ESTADO_CIVIL,DS_FAIXA_ETARIA,DS_GRAU_ESCOLARIDADE,DS_COR_RACA,QT_APTOS,QT_COMPARECIMENTO,QT_ABSTENCAO
0,PR,GODOY MOREIRA,MASCULINO,SOLTEIRO,16 anos,ENSINO MÉDIO INCOMPLETO,NÃO INFORMADO,9,8,1
1,PR,GODOY MOREIRA,MASCULINO,SOLTEIRO,17 anos,ENSINO FUNDAMENTAL INCOMPLETO,NÃO INFORMADO,1,1,0
2,PR,GODOY MOREIRA,MASCULINO,SOLTEIRO,18 anos,ENSINO FUNDAMENTAL INCOMPLETO,NÃO INFORMADO,3,2,1
3,PR,GODOY MOREIRA,MASCULINO,SOLTEIRO,19 anos,ENSINO MÉDIO INCOMPLETO,NÃO INFORMADO,14,11,3
4,PR,GODOY MOREIRA,MASCULINO,SOLTEIRO,19 anos,ENSINO MÉDIO COMPLETO,NÃO INFORMADO,4,3,1


In [15]:
# Diagnóstico inicial do TSE: dtypes e completude por coluna.
df_tse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8785738 entries, 0 to 8785737
Data columns (total 10 columns):
 #   Column                Dtype 
---  ------                ----- 
 0   SG_UF                 object
 1   NM_MUNICIPIO          object
 2   DS_GENERO             object
 3   DS_ESTADO_CIVIL       object
 4   DS_FAIXA_ETARIA       object
 5   DS_GRAU_ESCOLARIDADE  object
 6   DS_COR_RACA           object
 7   QT_APTOS              int64 
 8   QT_COMPARECIMENTO     int64 
 9   QT_ABSTENCAO          int64 
dtypes: int64(3), object(7)
memory usage: 670.3+ MB


## 6. TSE — Limpeza

A base é a granularidade máxima do TSE: cada linha é uma combinação (UF × município × zona × perfil demográfico). Aplicamos três tratamentos:

1. Remoção de linhas com `QT_APTOS == 0` (combinação demográfica sem eleitores — não contribui para taxas).
2. Padronização de strings (trim + uppercase em UF).
3. Conversão de categóricas para `category` (economia de memória).

In [16]:
# Remove combinações sem eleitores aptos.
# Essas linhas não contribuem para taxas (dividiriam por zero) e só poluem agregações.
linhas_antes = len(df_tse)
df_tse = df_tse[df_tse['QT_APTOS'] > 0].copy()
print(f"Linhas removidas (QT_APTOS=0): {linhas_antes - len(df_tse):,}")
print(f"Linhas restantes: {len(df_tse):,}")

# Normaliza espaços em colunas textuais; o TSE costuma trazer espaços extras em alguns campos.
colunas_texto = ['SG_UF', 'NM_MUNICIPIO', 'DS_GENERO', 'DS_ESTADO_CIVIL',
                 'DS_FAIXA_ETARIA', 'DS_GRAU_ESCOLARIDADE', 'DS_COR_RACA']
for col in colunas_texto:
    df_tse[col] = df_tse[col].astype(str).str.strip()

# UF para uppercase como defesa contra variações na origem.
df_tse['SG_UF'] = df_tse['SG_UF'].str.upper()

# Converte categóricas para category — drástica redução de memória dado o tamanho da base.
for col in colunas_texto:
    df_tse[col] = df_tse[col].astype('category')

df_tse.dtypes

Linhas removidas (QT_APTOS=0): 0
Linhas restantes: 8,785,738


SG_UF                   category
NM_MUNICIPIO            category
DS_GENERO               category
DS_ESTADO_CIVIL         category
DS_FAIXA_ETARIA         category
DS_GRAU_ESCOLARIDADE    category
DS_COR_RACA             category
QT_APTOS                   int64
QT_COMPARECIMENTO          int64
QT_ABSTENCAO               int64
dtype: object

## 7. TSE — Variáveis derivadas

Criamos:
- `TAXA_COMPARECIMENTO`: `QT_COMPARECIMENTO / QT_APTOS`
- `TAXA_ABSTENCAO`: `QT_ABSTENCAO / QT_APTOS`
- `REGIAO`: derivada da UF, replicando a categorização da CESOP para viabilizar o cruzamento geográfico.

In [17]:
# Taxas de comparecimento e abstenção: métricas centrais do TSE para o cruzamento com CESOP.
df_tse['TAXA_COMPARECIMENTO'] = df_tse['QT_COMPARECIMENTO'] / df_tse['QT_APTOS']
df_tse['TAXA_ABSTENCAO'] = df_tse['QT_ABSTENCAO'] / df_tse['QT_APTOS']

# REGIAO derivada da UF: replica a categorização do IBGE/CESOP para viabilizar o join geográfico.
uf_para_regiao = {
    'AC': 'Norte', 'AM': 'Norte', 'AP': 'Norte', 'PA': 'Norte',
    'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste',
    'PB': 'Nordeste', 'PE': 'Nordeste', 'PI': 'Nordeste', 'RN': 'Nordeste',
    'SE': 'Nordeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MS': 'Centro-Oeste',
    'MT': 'Centro-Oeste', 'ZZ': 'Exterior',
}
df_tse['REGIAO'] = df_tse['SG_UF'].astype(str).map(uf_para_regiao).astype('category')

df_tse[['SG_UF', 'REGIAO', 'TAXA_COMPARECIMENTO', 'TAXA_ABSTENCAO']].head()

,SG_UF,REGIAO,TAXA_COMPARECIMENTO,TAXA_ABSTENCAO
0,PR,Sul,0.888889,0.111111
1,PR,Sul,1.000000,0.000000
2,PR,Sul,0.666667,0.333333
3,PR,Sul,0.785714,0.214286
4,PR,Sul,0.750000,0.250000


## 8. TSE — Agregações

Dois recortes que serão usados nas análises:

- `df_tse_uf`: total de aptos/comparecimento/abstenção por UF → permite ranquear estados.
- `df_tse_perfil`: agregação por perfil demográfico (gênero × faixa etária × escolaridade × raça) → permite comparar com o perfil da amostra CESOP.

In [18]:
# Agregação por UF: soma dos contingentes e taxas globais.
# A taxa por UF é calculada a partir das somas (não como média das taxas-linha)
# para que zonas pequenas não distorçam o indicador estadual.
df_tse_uf = (
    df_tse
    .groupby('SG_UF', observed=True)
    .agg(
        QT_APTOS=('QT_APTOS', 'sum'),
        QT_COMPARECIMENTO=('QT_COMPARECIMENTO', 'sum'),
        QT_ABSTENCAO=('QT_ABSTENCAO', 'sum'),
    )
    .reset_index()
)
df_tse_uf['TAXA_COMPARECIMENTO'] = df_tse_uf['QT_COMPARECIMENTO'] / df_tse_uf['QT_APTOS']
df_tse_uf['TAXA_ABSTENCAO'] = df_tse_uf['QT_ABSTENCAO'] / df_tse_uf['QT_APTOS']
df_tse_uf['REGIAO'] = df_tse_uf['SG_UF'].astype(str).map(uf_para_regiao)
df_tse_uf.head()

,SG_UF,QT_APTOS,QT_COMPARECIMENTO,QT_ABSTENCAO,TAXA_COMPARECIMENTO,TAXA_ABSTENCAO,REGIAO
0,AC,1176866,877880,298986,0.745947,0.254053,Norte
1,AL,4651312,3590717,1060595,0.771979,0.228021,Nordeste
2,AM,5295496,4184165,1111331,0.790137,0.209863,Norte
3,AP,1101374,844475,256899,0.766747,0.233253,Norte
4,BA,22583056,17862306,4720750,0.790961,0.209039,Nordeste


In [19]:
# Agregação por perfil demográfico nacional.
# Comparar a composição da amostra CESOP com a população eleitoral TSE exige
# que as duas bases estejam no mesmo grão; este agregado é a referência TSE.
df_tse_perfil = (
    df_tse
    .groupby(
        ['DS_GENERO', 'DS_FAIXA_ETARIA', 'DS_GRAU_ESCOLARIDADE', 'DS_COR_RACA'],
        observed=True,
    )
    .agg(
        QT_APTOS=('QT_APTOS', 'sum'),
        QT_COMPARECIMENTO=('QT_COMPARECIMENTO', 'sum'),
        QT_ABSTENCAO=('QT_ABSTENCAO', 'sum'),
    )
    .reset_index()
)
df_tse_perfil['TAXA_COMPARECIMENTO'] = df_tse_perfil['QT_COMPARECIMENTO'] / df_tse_perfil['QT_APTOS']
df_tse_perfil['TAXA_ABSTENCAO'] = df_tse_perfil['QT_ABSTENCAO'] / df_tse_perfil['QT_APTOS']
df_tse_perfil.head()

,DS_GENERO,DS_FAIXA_ETARIA,DS_GRAU_ESCOLARIDADE,DS_COR_RACA,QT_APTOS,QT_COMPARECIMENTO,QT_ABSTENCAO,TAXA_COMPARECIMENTO,TAXA_ABSTENCAO
0,FEMININO,100 anos ou mais,ANALFABETO,NÃO INFORMADO,51234,538,50696,0.010501,0.989499
1,FEMININO,100 anos ou mais,ENSINO FUNDAMENTAL COMPLETO,NÃO INFORMADO,15790,176,15614,0.011146,0.988854
2,FEMININO,100 anos ou mais,ENSINO FUNDAMENTAL INCOMPLETO,NÃO INFORMADO,36604,382,36222,0.010436,0.989564
3,FEMININO,100 anos ou mais,ENSINO MÉDIO COMPLETO,NÃO INFORMADO,11646,188,11458,0.016143,0.983857
4,FEMININO,100 anos ou mais,ENSINO MÉDIO INCOMPLETO,NÃO INFORMADO,2390,70,2320,0.029289,0.970711


## 9. TSE — Validação e salvamento

Conferimos que as taxas estão em [0, 1] e que os shapes batem com o esperado. Salvamos três arquivos Parquet (linhas-detalhe + dois agregados).

In [20]:
# Sanidade: taxas devem ficar em [0, 1].
# Se passar, há bug no cálculo ou inconsistência da fonte (vale investigar antes de usar).
assert df_tse['TAXA_COMPARECIMENTO'].between(0, 1).all(), 'Taxas de comparecimento fora de [0, 1]'
assert df_tse['TAXA_ABSTENCAO'].between(0, 1).all(), 'Taxas de abstenção fora de [0, 1]'

print("=== df_tse (linhas-detalhe) ===")
print(df_tse.shape)
print("\n=== df_tse_uf (agregado por UF) ===")
print(df_tse_uf.shape)
print("\n=== df_tse_perfil (agregado por perfil) ===")
print(df_tse_perfil.shape)

=== df_tse (linhas-detalhe) ===
(8785738, 13)

=== df_tse_uf (agregado por UF) ===
(28, 7)

=== df_tse_perfil (agregado por perfil) ===
(498, 9)


In [21]:
# Salva os três recortes do TSE em Parquet.
# O detalhe é mantido por completude; os agregados são leves e práticos.
path_tse_clean = PATH_PROCESSED / 'tse_clean.parquet'
path_tse_uf = PATH_PROCESSED / 'tse_uf.parquet'
path_tse_perfil = PATH_PROCESSED / 'tse_perfil.parquet'

df_tse.to_parquet(path_tse_clean, index=False)
df_tse_uf.to_parquet(path_tse_uf, index=False)
df_tse_perfil.to_parquet(path_tse_perfil, index=False)

for p in (path_tse_clean, path_tse_uf, path_tse_perfil):
    print(f"Salvo: {p.name} ({p.stat().st_size / (1024**2):.2f} MB)")

Salvo: tse_clean.parquet (64.93 MB)
Salvo: tse_uf.parquet (0.01 MB)
Salvo: tse_perfil.parquet (0.02 MB)


## 10. Resumo do pipeline

Bases preparadas e salvas em `data/processed/`:

| Arquivo | Origem | O que contém |
|---|---|---|
| `cesop_clean.parquet` | CESOP `04832.SAV` | 2.000 respondentes com perguntas recodificadas, variáveis derivadas e dtypes ajustados |
| `tse_clean.parquet` | TSE BRASIL 2022 | Linhas-detalhe (município × zona × perfil) com taxas calculadas |
| `tse_uf.parquet` | Agregado | Totais e taxas por UF |
| `tse_perfil.parquet` | Agregado | Totais e taxas por perfil demográfico nacional |

**Próximo passo:** análises exploratórias nos notebooks `02_tratamento_dados.ipynb` e `03_eda_democracia.ipynb`.